# Políticas de decisión — demo

**Fase 3 · Tesis PR-196.** Compara tres políticas sobre el **mismo stream** (common random numbers):
`reglas` (baseline) · `multiseñal` · `LLM agéntico`.

Dos escenarios muestran las dos trampas: **covariate shift** (falsa alarma) y **concept drift** (punto ciego).

## Preparación

In [ ]:
import os, sys
from pathlib import Path

raiz = Path.cwd()
while not (raiz / "src").exists():
    raiz = raiz.parent
os.chdir(raiz)                       # rutas relativas (data/, models/) al proyecto
sys.path.insert(0, str(raiz / "src"))

import pandas as pd
from tesis.modelo_base import ModeloBase
from tesis.simulador.poblacion import cargar_poblacion, COHORTE_OOT
from tesis.simulador.simulador import ConfigSimulacion, plan_sano, agregar_episodio, Simulador
from tesis.senales.monitor import MonitorSenales, Referencia
from tesis.politicas import Contexto, crear, ClienteSimulado, PoliticaLLM
from tesis.experimentos.banco import comparar
print("proyecto:", raiz)

## Modelo víctima, población y referencia PSI

In [ ]:
modelo = ModeloBase.cargar("models/modelo_base.pkl")
pool = cargar_poblacion(cohorte=COHORTE_OOT)                 # futuro simulado (OOT)
dev = cargar_poblacion(cohorte=None)
dev = dev[dev["FECHA_CORTE"] < COHORTE_OOT]                 # train/DEV para la referencia
referencia = Referencia.construir(dev, modelo)

def politicas():
    return {
        "reglas": crear("reglas"),
        "multisenal": crear("multisenal"),
        "llm(sim)": PoliticaLLM(cliente=ClienteSimulado()),   # LLM simulado (gratis)
    }
print("pool:", pool.shape, "| dev:", dev.shape)

## Escenario A — covariate shift  (trampa: FALSA ALARMA)

La población cambia pero el modelo sigue ordenando bien el riesgo. El PSI sube → el baseline se asusta y reentrena de más.

In [ ]:
cfg_A = ConfigSimulacion(n_periodos=18, tam_lote=2000, semilla=13579)
plan_A = agregar_episodio(plan_sano(18), "covariate_shift", inicio=4, fin=11, intensidad=1.0, forma="rampa")
tabla_A, resumen_A = comparar(politicas(), pool, cfg_A, plan_A, modelo, referencia)
tabla_A

In [ ]:
resumen_A

## Escenario B — concept drift  (trampa: PUNTO CIEGO)

El modelo se rompe pero el PSI casi no se mueve: es invisible a los detectores de distribución. Solo el **AUC tardío** (12 meses de retardo) lo delata.

In [ ]:
cfg_B = ConfigSimulacion(n_periodos=24, tam_lote=2000, semilla=13579)
plan_B = agregar_episodio(plan_sano(24), "concept_drift", inicio=2, fin=11, intensidad=1.0, forma="escalon")
tabla_B, resumen_B = comparar(politicas(), pool, cfg_B, plan_B, modelo, referencia)
tabla_B

In [ ]:
resumen_B

## Traza REAL del LLM agéntico  (usa tu API key)

Corre el LLM real en 3 meses del escenario concept: sano → concept invisible → daño ya revelado.
Muestra **qué investiga** y **cómo justifica** su decisión.

> 💰 **Costo:** por defecto usa `claude-haiku-4-5` **sin razonamiento** → unos centavos (ideal para testear).
> Para la corrida final con mejor razonamiento, cambia a `claude-opus-5` (más caro).

In [ ]:
def recolectar(cfg, plan):
    mon = MonitorSenales(referencia)
    pares = []
    for lote in Simulador(pool, cfg, plan).stream():
        rep = mon.observar(lote, modelo)
        pares.append((rep, Contexto(historia=mon.reportes[:-1])))
    return pares

pares = recolectar(cfg_B, plan_B)
invisible = next(i for i, (r, _) in enumerate(pares)
                 if plan_B[r.periodo].tipo == "concept_drift" and r.auc_revelado is None)
danado = next(i for i, (r, _) in enumerate(pares)
              if r.auc_revelado is not None and r.auc_revelado < 0.5)

# TESTEO barato: modelo chico + sin thinking (centavos).
pol = PoliticaLLM(modelo="claude-haiku-4-5", pensar=False)
# Corrida final (mejor razonamiento, más caro):  pol = PoliticaLLM(modelo="claude-opus-5")

for i in [1, invisible, danado]:
    rep, ctx = pares[i]
    auc = f"{rep.auc_revelado:.3f}" if rep.auc_revelado is not None else "None (aún no madura)"
    print(f">>> Mes {rep.periodo} | tipo real={plan_B[rep.periodo].tipo} | psi={rep.psi_score:.3f} | auc_tardío={auc}")
    dec = pol.decidir(rep, ctx)
    tz = pol.trazas[-1]
    print("    herramientas:", [t["herramienta"] for t in tz["herramientas_usadas"]])
    print("    DECISIÓN:", dec.accion.value, "—", dec.razon, "\n")